Empiezo con una predicción lineal

In [1]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit # Para splits cronológicos múltiples
import time
import glob

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
N_LAGS = 24  # Aumentamos los lags, ya que predeciremos hasta 24h. Ajusta según EDA.
PREDICTION_HORIZONS = [1, 2, 4, 8, 12, 24] # Horizontes de predicción en horas
N_CV_SPLITS = 5 # Número de splits para TimeSeriesSplit (repeticiones de evaluación)
# TEST_SIZE_RATIO ya no se usa directamente, TimeSeriesSplit lo maneja.

# --- Rutas (Asegúrate de que coincidan con tu estructura) ---
PROJECT_ROOT_DIR = os.getcwd()

ORIGINAL_DATA_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs')
ORIGINAL_DATA_FILE = os.path.join(ORIGINAL_DATA_DIR, 'Air_Quality.csv')

SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')

INTERPOLATION_METHOD_NAME = 'interpolacionLineal'
IMPUTED_CSVS_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', INTERPOLATION_METHOD_NAME, 'CSVs')

PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'regresionLinealSimpleLags' # Mantener o cambiar si la estrategia es muy diferente
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)

CONTAMINANT_AND_AQI_COLS_IMPUTATION = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI', 'Date', 'City']


# --- Funciones Auxiliares ---

def create_lagged_features_for_horizon(df_series_input, target_col, n_lags, horizon):
    """
    Crea features de lags (hasta t-1) y el target y(t+h-1) -> y(t+horizon-1) para la estrategia directa.
    El índice de df_series_input se asume que es un RangeIndex (0 a N-1).
    """
    df_lagged = pd.DataFrame(index=df_series_input.index)
    
    # Crear Lags (features X)
    for i in range(1, n_lags + 1):
        df_lagged[f'{target_col}_lag{i}'] = df_series_input[target_col].shift(i)
        
    # Crear Target (y) para el horizonte especificado
    # y(t) será predicho usando X(t-1), X(t-2)...
    # Nuestro y real es AQI en el futuro. Si horizon=1, queremos predecir AQI(t) usando lags de AQI(t-1), AQI(t-2)...
    # El target y_t se alinea con los features X_t (que contienen lags del pasado de y_t)
    # Para predicción directa a h pasos: y_{t+h-1} = f(AQI_t-1, AQI_t-2, ...)
    # Pandas shift: shift(-k) adelanta los datos k periodos.
    # Queremos predecir `df_series_input[target_col]` en `horizon` pasos.
    # Entonces, el target es `df_series_input[target_col].shift(- (horizon -1) )` si los lags son de t.
    # Pero si nuestros lags son X_t = [y_{t-1}, y_{t-2}, ...], y queremos predecir y_{t+h-1},
    # el target es y.shift(-(horizon-1))
    # La forma más simple para estrategia directa: y es AQI.shift(-horizon) y los X son lags hasta el presente.
    
    df_lagged['target_h'] = df_series_input[target_col].shift(-horizon) # y_{t+h}
    
    # Eliminar NaNs generados por lags y por el shift del target
    df_lagged = df_lagged.dropna()
    
    if df_lagged.empty:
        return pd.DataFrame(), pd.Series(dtype='float64')

    y = df_lagged['target_h']
    X = df_lagged.drop(columns=['target_h'])
    
    return X, y


def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan # No se puede calcular diff
    d = np.abs(np.diff(y_train_for_naive)).sum() / (len(y_train_for_naive) - 1)
    if d == 0: return np.nan
    errors = np.abs(y_true - y_pred)
    return errors.mean() / d

def train_predict_eval_multi_horizon(df_city_data_input, target_col, n_lags, horizons_list, n_cv_splits):
    """
    Entrena, predice y evalúa para múltiples horizontes usando TimeSeriesSplit.
    df_city_data_input debe ser un DataFrame con una sola columna (la serie temporal target_col)
    y con índice reseteado (0 a N-1).
    """
    results_by_horizon = []

    if len(df_city_data_input) < n_lags + max(horizons_list) + n_cv_splits + 1:
        print(f"    Datos insuficientes para los horizontes y splits dados. Min requerido: {n_lags + max(horizons_list) + n_cv_splits +1}, disponible: {len(df_city_data_input)}")
        for h in horizons_list:
            results_by_horizon.append({'Horizon': h, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon

    for h_idx, horizon in enumerate(horizons_list):
        # print(f"      Horizonte: {horizon}h")
        X, y = create_lagged_features_for_horizon(df_city_data_input, target_col, n_lags, horizon)

        if X.empty or y.empty or len(X) < n_cv_splits * 2 : # Mínimo datos para TimeSeriesSplit
            # print(f"        Datos insuficientes tras crear lags/target para horizonte {horizon}h. X: {len(X)}, y: {len(y)}")
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue

        tscv = TimeSeriesSplit(n_splits=n_cv_splits)
        horizon_maes, horizon_rmses, horizon_mases = [], [], []

        for fold, (train_index, test_index) in enumerate(tscv.split(X)):
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]

            if X_train.empty or y_train.empty or X_test.empty or y_test.empty or len(y_train) <=1:
                # print(f"        Fold {fold+1}: Train o Test data vacíos o y_train muy corto. Omitiendo fold.")
                continue

            model = LinearRegression()
            try:
                model.fit(X_train, y_train)
                predictions = model.predict(X_test)

                mae = mean_absolute_error(y_test, predictions)
                rmse = np.sqrt(mean_squared_error(y_test, predictions))
                # Usar y_train de este fold para el MASE
                mase = calculate_mase(y_test.values, predictions, y_train.values)

                horizon_maes.append(mae)
                horizon_rmses.append(rmse)
                if not np.isnan(mase): horizon_mases.append(mase)
            except Exception as e:
                print(f"        Error en Fold {fold+1} para horizonte {horizon}h: {e}")
                continue
        
        results_by_horizon.append({
            'Horizon': horizon,
            'MAE': np.nanmean(horizon_maes) if horizon_maes else np.nan,
            'RMSE': np.nanmean(horizon_rmses) if horizon_rmses else np.nan,
            'MASE': np.nanmean(horizon_mases) if horizon_mases else np.nan
        })
    return results_by_horizon


# --- Script Principal ---
all_prediction_metrics_detailed = [] # Para guardar todas las repeticiones de CV y archivos
script_start_time = time.time()

# 1. Cargar DataFrame Original
try:
    df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    if 'CO2' in df_original_full.columns: df_original_full = df_original_full.drop(columns=['CO2'])
    print(f"DataFrame original '{ORIGINAL_DATA_FILE}' cargado.")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo original '{ORIGINAL_DATA_FILE}'. Omitiendo predicción con datos originales.")
    df_original_full = None

# Procesar DataFrame Original
if df_original_full is not None:
    print("\n--- Predicción con Datos Originales (sin NaNs simulados) ---")
    for city_name in df_original_full['City'].unique():
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        print(f"  Ciudad: {city_name}")
        # Trabajar con copia y resetear índice para consistencia con create_lagged_features_for_horizon
        df_city_original_series = df_original_full[df_original_full['City'] == city_name][[TARGET_VARIABLE]].copy().reset_index(drop=True)
        
        if df_city_original_series[TARGET_VARIABLE].isnull().any():
            # print(f"    Advertencia: NaNs en AQI para {city_name} (Original). Rellenando con ffill/bfill.")
            df_city_original_series[TARGET_VARIABLE] = df_city_original_series[TARGET_VARIABLE].ffill().bfill()
            if df_city_original_series[TARGET_VARIABLE].isnull().any():
                print(f"    No se pudieron rellenar NaNs en AQI para {city_name} (Original). Omitiendo.")
                continue
        
        horizon_metrics = train_predict_eval_multi_horizon(
            df_city_original_series, TARGET_VARIABLE, N_LAGS, PREDICTION_HORIZONS, N_CV_SPLITS
        )
        for hm in horizon_metrics:
            all_prediction_metrics_detailed.append({
                'Dataset_Type': 'Original', 'City': city_name, 'Scenario': 'N/A',
                'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'],
                'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']
            })
        # Imprimir resumen para el primer y último horizonte como ejemplo
        if horizon_metrics:
            print(f"    Métricas (Original, {horizon_metrics[0]['Horizon']}h): MAE={horizon_metrics[0]['MAE']:.4f}, RMSE={horizon_metrics[0]['RMSE']:.4f}, MASE={horizon_metrics[0]['MASE']:.4f}")
            if len(horizon_metrics) > 1:
                 print(f"    Métricas (Original, {horizon_metrics[-1]['Horizon']}h): MAE={horizon_metrics[-1]['MAE']:.4f}, RMSE={horizon_metrics[-1]['RMSE']:.4f}, MASE={horizon_metrics[-1]['MASE']:.4f}")


datasets_to_process_info = [
    ("Simulated_NaNs", SIMULATED_DATA_ROOT_DIR, "_with_nans.csv"),
    ("Imputed_Linear", IMPUTED_CSVS_ROOT_DIR, "_imputed_linear.csv")
]

for dataset_type, base_path, file_suffix in datasets_to_process_info:
    print(f"\n--- Predicción con Datos: {dataset_type} ---")
    if not os.path.exists(base_path):
        print(f"Directorio base no encontrado para {dataset_type}: {base_path}. Omitiendo.")
        continue

    for city_folder_name_safe in os.listdir(base_path):
        city_path = os.path.join(base_path, city_folder_name_safe)
        if not os.path.isdir(city_path): continue
        
        original_city_name = city_folder_name_safe.replace("_", " ") # Asumimos que el nombre de carpeta es el "safe"
        print(f"  Ciudad: {original_city_name} (desde carpeta: {city_folder_name_safe})")

        for scenario_folder_name_safe in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder_name_safe)
            if not os.path.isdir(scenario_path): continue
            print(f"    Escenario: {scenario_folder_name_safe}")

            pattern = os.path.join(scenario_path, f"*{file_suffix}")
            repetition_files = glob.glob(pattern)
            if not repetition_files: continue

            # Métricas para promediar sobre las repeticiones de este escenario/ciudad/horizonte
            # Estructura: {horizon: {'maes': [], 'rmses': [], 'mases': []}}
            scenario_horizon_metrics_accumulator = {h: {'maes': [], 'rmses': [], 'mases': []} for h in PREDICTION_HORIZONS}

            for rep_file_path in repetition_files:
                file_name = os.path.basename(rep_file_path)
                try:
                    # Extraer ID de repetición (ej. rep1_seed123)
                    rep_file_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                    df_rep = pd.read_csv(rep_file_path, usecols=lambda c: c in CONTAMINANT_AND_AQI_COLS_IMPUTATION)
                    df_rep['Date'] = pd.to_datetime(df_rep['Date'])
                except Exception as e:
                    print(f"        Error cargando/procesando archivo {rep_file_path}: {e}. Omitiendo.")
                    continue

                # Usar el nombre de ciudad del archivo si es diferente al de la carpeta (poco probable aquí)
                # current_city_in_file = df_rep['City'].unique()[0] if 'City' in df_rep.columns else original_city_name
                
                df_city_rep_series = df_rep[df_rep['City'] == original_city_name][[TARGET_VARIABLE]].copy().reset_index(drop=True)

                if dataset_type == "Simulated_NaNs" and df_city_rep_series[TARGET_VARIABLE].isnull().any():
                    df_city_rep_series[TARGET_VARIABLE] = df_city_rep_series[TARGET_VARIABLE].ffill().bfill()
                    if df_city_rep_series[TARGET_VARIABLE].isnull().any():
                        # print(f"        No se pudieron rellenar NaNs en {original_city_name} (Simulated_NaNs) para {file_name}. Omitiendo archivo.")
                        continue
                
                horizon_metrics_rep = train_predict_eval_multi_horizon(
                    df_city_rep_series, TARGET_VARIABLE, N_LAGS, PREDICTION_HORIZONS, N_CV_SPLITS
                )

                for hm_rep in horizon_metrics_rep:
                    h = hm_rep['Horizon']
                    # Guardar métrica detallada de esta repetición y split de CV (implícito en el promedio de hm_rep)
                    all_prediction_metrics_detailed.append({
                        'Dataset_Type': dataset_type, 'City': original_city_name, 'Scenario': scenario_folder_name_safe,
                        'Repetition_File_ID': rep_file_id, 'Horizon_h': h,
                        'MAE': hm_rep['MAE'], 'RMSE': hm_rep['RMSE'], 'MASE': hm_rep['MASE']
                    })
                    # Acumular para el promedio del escenario/horizonte
                    if not np.isnan(hm_rep['MAE']): scenario_horizon_metrics_accumulator[h]['maes'].append(hm_rep['MAE'])
                    if not np.isnan(hm_rep['RMSE']): scenario_horizon_metrics_accumulator[h]['rmses'].append(hm_rep['RMSE'])
                    if not np.isnan(hm_rep['MASE']): scenario_horizon_metrics_accumulator[h]['mases'].append(hm_rep['MASE'])
            
            # Calcular y mostrar promedios para el escenario y algunos horizontes
            print(f"      Promedios para Escenario '{scenario_folder_name_safe}':")
            for h_summary in [PREDICTION_HORIZONS[0], PREDICTION_HORIZONS[-1]]: # Mostrar primero y último horizonte
                 if h_summary in scenario_horizon_metrics_accumulator:
                    avg_s_mae = np.nanmean(scenario_horizon_metrics_accumulator[h_summary]['maes'])
                    avg_s_rmse = np.nanmean(scenario_horizon_metrics_accumulator[h_summary]['rmses'])
                    avg_s_mase = np.nanmean(scenario_horizon_metrics_accumulator[h_summary]['mases'])
                    print(f"        Horizonte {h_summary}h: MAE={avg_s_mae:.4f}, RMSE={avg_s_rmse:.4f}, MASE={avg_s_mase:.4f}")


# --- Guardado de Métricas ---
df_all_prediction_metrics_detailed = pd.DataFrame(all_prediction_metrics_detailed)

# 1. Guardar el CSV detallado global
if not df_all_prediction_metrics_detailed.empty:
    detailed_output_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_prediction_metrics_detailed.to_csv(detailed_output_path, index=False)
    print(f"\nTodas las métricas de predicción detalladas guardadas en: {detailed_output_path}")
    print(df_all_prediction_metrics_detailed.head())

# 2. Crear y guardar CSV de métricas promedio por ciudad
# Agrupar por Ciudad, Dataset_Type, Scenario, Horizon_h y calcular el promedio de MAE, RMSE, MASE
if not df_all_prediction_metrics_detailed.empty:
    df_summary_by_city_scenario_horizon = df_all_prediction_metrics_detailed.groupby(
        ['City', 'Dataset_Type', 'Scenario', 'Horizon_h']
    ).agg(
        Avg_MAE=('MAE', 'mean'),
        Avg_RMSE=('RMSE', 'mean'),
        Avg_MASE=('MASE', 'mean'),
        Std_MAE=('MAE', 'std'), # Opcional: Desviación estándar
        Std_RMSE=('RMSE', 'std'),
        Std_MASE=('MASE', 'std'),
        Count_Repetitions=('MAE', 'count') # Para verificar cuántas repeticiones contribuyeron
    ).reset_index()

    for city_name_original_unique in df_summary_by_city_scenario_horizon['City'].unique():
        city_name_s = city_name_original_unique.replace(" ", "_").replace("-", "_")
        df_city_summary = df_summary_by_city_scenario_horizon[df_summary_by_city_scenario_horizon['City'] == city_name_original_unique]
        
        city_metrics_dir = os.path.join(PREDICTION_METRICS_ROOT_DIR, city_name_s) # Nueva subcarpeta por ciudad
        os.makedirs(city_metrics_dir, exist_ok=True)
        city_summary_file_path = os.path.join(city_metrics_dir, f'metrics_summary_{PREDICTION_METHOD_NAME}_{city_name_s}.csv')
        
        df_city_summary.to_csv(city_summary_file_path, index=False)
        print(f"  Resumen de métricas para {city_name_original_unique} guardado en: {city_summary_file_path}")

# 3. Crear y guardar CSV de métricas globales promediadas por Dataset_Type y Horizon_h (sobre todas las ciudades y escenarios)
if not df_all_prediction_metrics_detailed.empty:
    df_global_summary_by_dataset_horizon = df_all_prediction_metrics_detailed.groupby(
        ['Dataset_Type', 'Horizon_h'] # Podrías añadir 'Scenario' aquí si quieres ver el promedio por escenario globalmente
    ).agg(
        Global_Avg_MAE=('MAE', 'mean'),
        Global_Avg_RMSE=('RMSE', 'mean'),
        Global_Avg_MASE=('MASE', 'mean')
    ).reset_index()
    
    global_summary_file_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_global_summary_{PREDICTION_METHOD_NAME}.csv')
    df_global_summary_by_dataset_horizon.to_csv(global_summary_file_path, index=False)
    print(f"\nResumen global de métricas promedio por tipo de dataset y horizonte guardado en: {global_summary_file_path}")
    print(df_global_summary_by_dataset_horizon)


script_end_time = time.time()
print(f"\nScript de predicción completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")

DataFrame original 'c:\Users\fdemiguel\Documentos\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargado.

--- Predicción con Datos Originales (sin NaNs simulados) ---
  Ciudad: Brasilia
    Métricas (Original, 1h): MAE=1.8579, RMSE=2.6920, MASE=1.1286
    Métricas (Original, 24h): MAE=3.8425, RMSE=5.3649, MASE=2.3295
  Ciudad: Cairo
    Métricas (Original, 1h): MAE=1.9158, RMSE=3.2011, MASE=1.9336
    Métricas (Original, 24h): MAE=9.7798, RMSE=12.2245, MASE=10.1662
  Ciudad: Dubai
    Métricas (Original, 1h): MAE=0.8385, RMSE=1.5415, MASE=0.9591
    Métricas (Original, 24h): MAE=13.0547, RMSE=17.4330, MASE=14.8770
  Ciudad: London
    Métricas (Original, 1h): MAE=2.0932, RMSE=2.8293, MASE=1.7445
    Métricas (Original, 24h): MAE=6.0932, RMSE=8.3926, MASE=5.0437
  Ciudad: New York
    Métricas (Original, 1h): MAE=2.7261, RMSE=4.0730, MASE=1.6947
    Métricas (Original, 24h): MAE=9.1634, RMSE=11.8341, MASE=5.6096
  Ciudad: Sydney
    Métricas (Original, 1h): MAE=1.8503, RMSE=2.8655, MA

SARIMA

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit # Para validación cruzada cronológica
import time
import glob
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning) # Para statsmodels

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE_SARIMA = 'AQI' # La variable que queremos predecir
PREDICTION_HORIZONS_SARIMA = [1, 2, 4, 8, 12, 24] # Horizontes de predicción en horas
N_CV_SPLITS_SARIMA = 5 # Número de splits para TimeSeriesSplit
COLUMNS_TO_LOAD_FROM_CSV = ['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'] # Asegurar que AQI esté

# --- Rutas ---
PROJECT_ROOT_DIR_SARIMA_PRED = os.getcwd()

# Input: Datos originales y datos imputados
ORIGINAL_DATA_DIR_SARIMA = os.path.join(PROJECT_ROOT_DIR_SARIMA_PRED, 'CSVs')
ORIGINAL_DATA_FILE_SARIMA = os.path.join(ORIGINAL_DATA_DIR_SARIMA, 'Air_Quality.csv')

SIMULATED_DATA_ROOT_DIR_SARIMA = os.path.join(PROJECT_ROOT_DIR_SARIMA_PRED, 'CSVs', 'simulados') # Datos con NaNs simulados (para referencia si es necesario)

# Asumimos que los datos imputados por diferentes métodos están en subcarpetas de 'imputaciones'
IMPUTATIONS_BASE_INPUT_DIR_SARIMA = os.path.join(PROJECT_ROOT_DIR_SARIMA_PRED, 'imputaciones')
# Ejemplo: si tienes datos imputados por interpolación lineal:
# IMPUTED_LINEAR_CSVS_ROOT_DIR = os.path.join(IMPUTATIONS_BASE_INPUT_DIR_SARIMA, 'interpolacionLineal', 'CSVs')
# Ejemplo: si tienes datos imputados por SARIMA (de la celda anterior):
# IMPUTED_SARIMA_CSVS_ROOT_DIR = os.path.join(IMPUTATIONS_BASE_INPUT_DIR_SARIMA, 'SARIMA_auto', 'CSVs')


# Input: Órdenes SARIMA guardados
ORDERS_JSON_INPUT_DIR_SARIMA = os.path.join(PROJECT_ROOT_DIR_SARIMA_PRED, 'imputaciones', 'SARIMA_auto', 'model_orders')

# Output: Métricas de predicción SARIMA
PREDICTION_METHOD_NAME_SARIMA = 'SARIMA_pred' # Nombre para la carpeta de este método de predicción
PREDICTION_METRICS_SARIMA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_PRED, 'predicciones', PREDICTION_METHOD_NAME_SARIMA, 'metricas')
os.makedirs(PREDICTION_METRICS_SARIMA_ROOT_DIR, exist_ok=True)


# --- Funciones Auxiliares ---
def calculate_mase_pred(y_true, y_pred, y_train_for_naive):
    """Calcula MASE para predicciones."""
    if len(y_train_for_naive) <= 1: return np.nan
    # Naive forecast: y_t = y_{t-1}
    # Los errores de la predicción ingenua en el conjunto de entrenamiento son |y_t - y_{t-1}|
    mae_naive_on_train = np.mean(np.abs(np.diff(y_train_for_naive)))
    if mae_naive_on_train == 0: return np.nan # Evitar división por cero
    
    actual_forecast_errors = np.abs(y_true - y_pred)
    mase = np.mean(actual_forecast_errors) / mae_naive_on_train
    return mase

def train_predict_eval_sarima_multi_horizon(
    df_series_target_input, # DataFrame con una columna: la serie del TARGET_VARIABLE_SARIMA, índice reseteado
    sarima_order, sarima_seasonal_order,
    horizons_list, n_cv_splits
):
    """
    Entrena un modelo SARIMA y evalúa predicciones para múltiples horizontes usando TimeSeriesSplit.
    """
    results_by_horizon_sarima = []
    series_data_for_sarima = df_series_target_input.iloc[:, 0] # Tomar la primera (y única) columna como pd.Series

    # Se necesita una longitud mínima para que TimeSeriesSplit funcione y para entrenar SARIMA.
    # El número de observaciones debe ser mayor que el número de parámetros.
    # min_obs_needed = sum(sarima_order) + sum(sarima_seasonal_order[:3]) + sarima_seasonal_order[3] + n_cv_splits + max(horizons_list)
    min_obs_needed = max(sum(sarima_order[:2]), sum(sarima_seasonal_order[:2])) + sarima_seasonal_order[3] if sarima_seasonal_order[3] > 0 else max(sum(sarima_order[:2]),1)
    min_obs_needed = max(min_obs_needed * (n_cv_splits +1) , n_cv_splits * 10, max(horizons_list) + 20) # Heurística

    if len(series_data_for_sarima) < min_obs_needed:
        # print(f"      Datos insuficientes ({len(series_data_for_sarima)}) para SARIMA con CV y horizontes. Mínimo heurístico: {min_obs_needed}")
        for h_s in horizons_list:
            results_by_horizon_sarima.append({'Horizon': h_s, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon_sarima

    tscv_sarima = TimeSeriesSplit(n_splits=n_cv_splits)
    
    # Para almacenar métricas de todos los folds para cada horizonte
    # {horizon: {'maes': [], 'rmses': [], 'mases': []}}
    accumulated_metrics_all_horizons = {h: {'maes': [], 'rmses': [], 'mases': []} for h in horizons_list}

    for fold, (train_idx_pos, test_idx_pos) in enumerate(tscv_sarima.split(series_data_for_sarima)):
        train_series = series_data_for_sarima.iloc[train_idx_pos]
        test_series = series_data_for_sarima.iloc[test_idx_pos]

        if len(train_series) < min_obs_needed / (n_cv_splits+1) or len(train_series) <=1 : # Asegurar que el train de este fold sea suficiente
            # print(f"        Fold {fold+1}: Train data muy corto ({len(train_series)}). Omitiendo fold.")
            continue
        if len(test_series) < max(horizons_list):
             # print(f"        Fold {fold+1}: Test data más corto ({len(test_series)}) que el horizonte máximo ({max(horizons_list)}). Ajustando test o omitiendo.")
             # Podríamos truncar horizontes o simplemente omitir el fold si es problemático.
             # Por ahora, continuaremos, pero las predicciones para horizontes largos fallarán.
             pass


        try:
            model_sarima = SARIMAX(train_series, order=sarima_order, seasonal_order=sarima_seasonal_order,
                                   enforce_stationarity=False, enforce_invertibility=False,
                                   initialization='approximate_diffuse')
            results_sarima = model_sarima.fit(disp=False, maxiter=150) # Reducir maxiter para velocidad

            # Generar predicciones para todos los horizontes necesarios para este fold
            # El número de pasos a predecir es la longitud del conjunto de test
            max_steps_to_forecast_for_fold = len(test_series)
            
            if max_steps_to_forecast_for_fold > 0:
                forecasts_all_steps = results_sarima.forecast(steps=max_steps_to_forecast_for_fold)
            else:
                forecasts_all_steps = pd.Series(dtype='float64')


            for h_s in horizons_list:
                if h_s <= max_steps_to_forecast_for_fold:
                    y_true_h = test_series.iloc[:h_s] # Ground truth para los primeros h_s pasos del test set
                    y_pred_h = forecasts_all_steps.iloc[:h_s] # Predicciones para los primeros h_s pasos

                    # Asegurar que y_true_h y y_pred_h tengan la misma longitud para la métrica
                    # Esto es importante si forecast devuelve menos de h_s pasos o si test_series es más corto.
                    common_len = min(len(y_true_h), len(y_pred_h))
                    if common_len == 0: continue

                    y_true_h_common = y_true_h.iloc[:common_len].values
                    y_pred_h_common = y_pred_h.iloc[:common_len].values
                    
                    # Para MASE, necesitamos y_true hasta el final del horizonte h_s-1
                    # y_pred hasta h_s-1. Usamos el último valor de y_pred y y_true para el error del paso h.
                    # Esto se complica. Para predicción directa en h, y_true es y_{t+h} y y_pred es y_hat_{t+h|t}
                    # Aquí, y_true_h_common es [y_{test_start}, ..., y_{test_start+h-1}]
                    # y_pred_h_common es [y_hat_{test_start|train_end}, ..., y_hat_{test_start+h-1|train_end}]
                    # Para MASE, el error es sobre el paso específico.
                    # Tomamos el error del último paso del horizonte.
                    
                    if common_len > 0:
                        mae_h = mean_absolute_error(y_true_h_common, y_pred_h_common)
                        rmse_h = np.sqrt(mean_squared_error(y_true_h_common, y_pred_h_common))
                        # Para MASE, usamos el error de la predicción del último paso del horizonte actual
                        # y_true_h_step = y_true_h_common[-1:]
                        # y_pred_h_step = y_pred_h_common[-1:]
                        # mase_h = calculate_mase_pred(y_true_h_step, y_pred_h_step, train_series.values)
                        # O, más comúnmente para multi-step, se promedian los errores escalados.
                        # Aquí, calculamos MASE sobre el error promedio de los h pasos
                        mase_h = calculate_mase_pred(y_true_h_common, y_pred_h_common, train_series.values)


                        accumulated_metrics_all_horizons[h_s]['maes'].append(mae_h)
                        accumulated_metrics_all_horizons[h_s]['rmses'].append(rmse_h)
                        if not np.isnan(mase_h):
                            accumulated_metrics_all_horizons[h_s]['mases'].append(mase_h)
                # else:
                    # print(f"        Fold {fold+1}: Horizonte {h_s}h excede la longitud del test set disponible ({max_steps_to_forecast_for_fold}).")

        except Exception as e_sarima_fit:
            print(f"        Error en Fold {fold+1} durante ajuste/predicción SARIMA: {e_sarima_fit}")
            # No se añaden métricas para este fold si falla
            continue
            
    # Calcular promedios finales para cada horizonte sobre los folds exitosos
    for h_s in horizons_list:
        results_by_horizon_sarima.append({
            'Horizon': h_s,
            'MAE': np.nanmean(accumulated_metrics_all_horizons[h_s]['maes']) if accumulated_metrics_all_horizons[h_s]['maes'] else np.nan,
            'RMSE': np.nanmean(accumulated_metrics_all_horizons[h_s]['rmses']) if accumulated_metrics_all_horizons[h_s]['rmses'] else np.nan,
            'MASE': np.nanmean(accumulated_metrics_all_horizons[h_s]['mases']) if accumulated_metrics_all_horizons[h_s]['mases'] else np.nan
        })
        
    return results_by_horizon_sarima


# --- Script Principal de Predicción SARIMA ---
all_prediction_metrics_sarima_detailed = []
script_start_time_sarima_pred = time.time()

# 1. Cargar DataFrame Original (Ground Truth para limpieza, si es necesario)
try:
    df_original_full_sarima = pd.read_csv(ORIGINAL_DATA_FILE_SARIMA)
    df_original_full_sarima['Date'] = pd.to_datetime(df_original_full_sarima['Date'])
    if 'CO2' in df_original_full_sarima.columns: df_original_full_sarima = df_original_full_sarima.drop(columns=['CO2'])
    print(f"DataFrame original '{ORIGINAL_DATA_FILE_SARIMA}' cargado para predicción SARIMA.")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo original '{ORIGINAL_DATA_FILE_SARIMA}'. No se pueden obtener datos limpios base.")
    df_original_full_sarima = None # Continuar sin él si no se usa para rellenar datos de entrada

# Definir los tipos de datasets de entrada para la predicción
# El primer elemento del tuple es el nombre descriptivo
# El segundo es la ruta raíz a los CSVs (que contienen subcarpetas por ciudad, escenario, rep)
# El tercero es el sufijo común de los archivos en la carpeta de repetición
datasets_to_predict_with_sarima = [
    ("Original_Clean", ORIGINAL_DATA_DIR_SARIMA, ".csv"), # Tratar el original como un "dataset"
    # Añade aquí las rutas a tus datos imputados por otros métodos
    ("Imputed_LinearInterpolation", os.path.join(IMPUTATIONS_BASE_INPUT_DIR_SARIMA, 'interpolacionLineal', 'CSVs'), "_imputed_linear.csv"),
    ("Imputed_SARIMA_auto", os.path.join(IMPUTATIONS_BASE_INPUT_DIR_SARIMA, 'SARIMA_auto', 'CSVs'), "_imputed_sarima.csv"),
    # ("Simulated_NaNs", SIMULATED_DATA_ROOT_DIR_SARIMA, "_with_nans.csv"), # Podrías incluir esto si quieres ver cómo SARIMA maneja NaNs directamente (no recomendado sin pre-imputación)
]


for dataset_type_sarima, base_path_sarima, file_suffix_sarima in datasets_to_predict_with_sarima:
    print(f"\n--- Predicción SARIMA con Datos: {dataset_type_sarima} ---")
    if not os.path.exists(base_path_sarima):
        print(f"Directorio base no encontrado para {dataset_type_sarima}: {base_path_sarima}. Omitiendo.")
        continue

    # Lógica diferente para "Original_Clean" ya que no tiene estructura de escenario/repetición
    if dataset_type_sarima == "Original_Clean":
        if df_original_full_sarima is None: continue
        for city_name_orig_s in df_original_full_sarima['City'].unique():
            city_name_safe_s = city_name_orig_s.replace(" ", "_").replace("-", "_")
            print(f"  Ciudad: {city_name_orig_s} (Dataset: {dataset_type_sarima})")
            
            df_city_data_orig_s = df_original_full_sarima[df_original_full_sarima['City'] == city_name_orig_s][[TARGET_VARIABLE_SARIMA]].copy().reset_index(drop=True)
            
            # Rellenar NaNs si existen en el original (aunque idealmente no deberían para "Clean")
            if df_city_data_orig_s[TARGET_VARIABLE_SARIMA].isnull().any():
                df_city_data_orig_s[TARGET_VARIABLE_SARIMA] = df_city_data_orig_s[TARGET_VARIABLE_SARIMA].ffill().bfill()
            if df_city_data_orig_s[TARGET_VARIABLE_SARIMA].isnull().any(): # Si aún hay NaNs
                print(f"    No se pudieron rellenar NaNs en {TARGET_VARIABLE_SARIMA} para {city_name_orig_s} (Original). Omitiendo.")
                continue

            # Cargar órdenes SARIMA para esta ciudad y variable objetivo
            order_file_path_s = os.path.join(ORDERS_JSON_INPUT_DIR_SARIMA, f'orders_{city_name_safe_s}_{TARGET_VARIABLE_SARIMA}.json')
            try:
                with open(order_file_path_s, 'r') as f: orders_s = json.load(f)
                order_tuple_s = tuple(orders_s['order']); seasonal_order_tuple_s = tuple(orders_s['seasonal_order'])
            except FileNotFoundError:
                print(f"    Archivo de órdenes no encontrado: {order_file_path_s}. Usando (1,0,1)(0,0,0,0) para {TARGET_VARIABLE_SARIMA}.")
                order_tuple_s = (1,0,1); seasonal_order_tuple_s = (0,0,0,0)
            except Exception as e_json_s:
                print(f"    Error cargando/parseando JSON {order_file_path_s}: {e_json_s}. Usando (1,0,1)(0,0,0,0).")
                order_tuple_s = (1,0,1); seasonal_order_tuple_s = (0,0,0,0)

            horizon_metrics_s = train_predict_eval_sarima_multi_horizon(
                df_city_data_orig_s, order_tuple_s, seasonal_order_tuple_s,
                PREDICTION_HORIZONS_SARIMA, N_CV_SPLITS_SARIMA
            )
            for hm_s in horizon_metrics_s:
                all_prediction_metrics_sarima_detailed.append({
                    'Dataset_Type': dataset_type_sarima, 'City': city_name_orig_s, 'Scenario': 'N/A_Original',
                    'Repetition_File_ID': 'N/A_Original', 'Horizon_h': hm_s['Horizon'],
                    'MAE': hm_s['MAE'], 'RMSE': hm_s['RMSE'], 'MASE': hm_s['MASE']
                })
            if horizon_metrics_s:
                print(f"    Métricas SARIMA ({dataset_type_sarima}, {horizon_metrics_s[0]['Horizon']}h): MAE={horizon_metrics_s[0]['MAE']:.4f}, RMSE={horizon_metrics_s[0]['RMSE']:.4f}, MASE={horizon_metrics_s[0]['MASE']:.4f}")
                if len(horizon_metrics_s) > 1: print(f"    Métricas SARIMA ({dataset_type_sarima}, {horizon_metrics_s[-1]['Horizon']}h): MAE={horizon_metrics_s[-1]['MAE']:.4f}, RMSE={horizon_metrics_s[-1]['RMSE']:.4f}, MASE={horizon_metrics_s[-1]['MASE']:.4f}")
        continue # Pasar al siguiente tipo de dataset

    # Para datasets imputados (que tienen la estructura ciudad/escenario/repetición)
    for city_folder_name_safe_s in os.listdir(base_path_sarima):
        city_path_s = os.path.join(base_path_sarima, city_folder_name_safe_s)
        if not os.path.isdir(city_path_s): continue
        
        original_city_name_s = city_folder_name_safe_s.replace("_", " ")
        print(f"  Ciudad: {original_city_name_s} (Dataset: {dataset_type_sarima}, desde carpeta: {city_folder_name_safe_s})")

        # Cargar órdenes SARIMA para esta ciudad y variable objetivo
        order_file_path_s = os.path.join(ORDERS_JSON_INPUT_DIR_SARIMA, f'orders_{city_folder_name_safe_s}_{TARGET_VARIABLE_SARIMA}.json')
        try:
            with open(order_file_path_s, 'r') as f: orders_s = json.load(f)
            order_tuple_s = tuple(orders_s['order']); seasonal_order_tuple_s = tuple(orders_s['seasonal_order'])
        except FileNotFoundError:
            print(f"    Archivo de órdenes no encontrado para {TARGET_VARIABLE_SARIMA}: {order_file_path_s}. Usando (1,0,1)(0,0,0,0). Omitiendo ciudad para este dataset.")
            continue # No se puede predecir sin órdenes
        except Exception as e_json_s:
            print(f"    Error cargando/parseando JSON {order_file_path_s}: {e_json_s}. Usando (1,0,1)(0,0,0,0). Omitiendo ciudad para este dataset.")
            continue

        for scenario_folder_name_safe_s in os.listdir(city_path_s):
            scenario_path_s = os.path.join(city_path_s, scenario_folder_name_safe_s)
            if not os.path.isdir(scenario_path_s): continue
            print(f"    Escenario: {scenario_folder_name_safe_s}")

            pattern_s = os.path.join(scenario_path_s, f"*{file_suffix_sarima}")
            repetition_files_s = sorted(glob.glob(pattern_s)) # Ordenar para consistencia
            if not repetition_files_s: continue

            scenario_horizon_metrics_accumulator_s = {h: {'maes': [], 'rmses': [], 'mases': []} for h in PREDICTION_HORIZONS_SARIMA}

            for rep_file_path_s in repetition_files_s:
                file_name_s = os.path.basename(rep_file_path_s)
                try:
                    rep_file_id_s = "_".join(file_name_s.split('_')[-4:-2]) if "_rep" in file_name_s and "_seed" in file_name_s else file_name_s
                    # Cargar solo las columnas necesarias
                    df_rep_s = pd.read_csv(rep_file_path_s, usecols=lambda c: c in COLUMNS_TO_LOAD_FROM_CSV)
                    # df_rep_s['Date'] = pd.to_datetime(df_rep_s['Date']) # Ya debería ser datetime si se guardó bien
                except Exception as e_load_s:
                    print(f"        Error cargando/procesando archivo {rep_file_path_s}: {e_load_s}. Omitiendo.")
                    continue
                
                # La serie para SARIMA es solo la columna TARGET_VARIABLE_SARIMA
                # Asegurar que el índice sea un RangeIndex simple para TimeSeriesSplit y el modelo
                df_city_rep_target_series_s = df_rep_s[df_rep_s['City'] == original_city_name_s][[TARGET_VARIABLE_SARIMA]].copy().reset_index(drop=True)

                # Los datos imputados no deberían tener NaNs en la columna objetivo. Si los tienen, es un problema.
                if df_city_rep_target_series_s[TARGET_VARIABLE_SARIMA].isnull().any():
                    print(f"        Advertencia: NaNs encontrados en {TARGET_VARIABLE_SARIMA} del archivo imputado {file_name_s}. Rellenando con ffill/bfill.")
                    df_city_rep_target_series_s[TARGET_VARIABLE_SARIMA] = df_city_rep_target_series_s[TARGET_VARIABLE_SARIMA].ffill().bfill()
                    if df_city_rep_target_series_s[TARGET_VARIABLE_SARIMA].isnull().any():
                         print(f"        No se pudieron rellenar NaNs en {TARGET_VARIABLE_SARIMA} para {file_name_s}. Omitiendo archivo.")
                         continue
                
                horizon_metrics_rep_s = train_predict_eval_sarima_multi_horizon(
                    df_city_rep_target_series_s, order_tuple_s, seasonal_order_tuple_s,
                    PREDICTION_HORIZONS_SARIMA, N_CV_SPLITS_SARIMA
                )

                for hm_rep_s in horizon_metrics_rep_s:
                    h_s_val = hm_rep_s['Horizon']
                    all_prediction_metrics_sarima_detailed.append({
                        'Dataset_Type': dataset_type_sarima, 'City': original_city_name_s, 'Scenario': scenario_folder_name_safe_s,
                        'Repetition_File_ID': rep_file_id_s, 'Horizon_h': h_s_val,
                        'MAE': hm_rep_s['MAE'], 'RMSE': hm_rep_s['RMSE'], 'MASE': hm_rep_s['MASE']
                    })
                    if not np.isnan(hm_rep_s['MAE']): scenario_horizon_metrics_accumulator_s[h_s_val]['maes'].append(hm_rep_s['MAE'])
                    if not np.isnan(hm_rep_s['RMSE']): scenario_horizon_metrics_accumulator_s[h_s_val]['rmses'].append(hm_rep_s['RMSE'])
                    if not np.isnan(hm_rep_s['MASE']): scenario_horizon_metrics_accumulator_s[h_s_val]['mases'].append(hm_rep_s['MASE'])
            
            print(f"      Promedios SARIMA para Escenario '{scenario_folder_name_safe_s}':")
            for h_summary_s in [PREDICTION_HORIZONS_SARIMA[0], PREDICTION_HORIZONS_SARIMA[-1]]:
                 if h_summary_s in scenario_horizon_metrics_accumulator_s:
                    avg_s_mae_s = np.nanmean(scenario_horizon_metrics_accumulator_s[h_summary_s]['maes'])
                    avg_s_rmse_s = np.nanmean(scenario_horizon_metrics_accumulator_s[h_summary_s]['rmses'])
                    avg_s_mase_s = np.nanmean(scenario_horizon_metrics_accumulator_s[h_summary_s]['mases'])
                    print(f"        Horizonte {h_summary_s}h: MAE={avg_s_mae_s:.4f}, RMSE={avg_s_rmse_s:.4f}, MASE={avg_s_mase_s:.4f}")


# --- Guardado de Métricas de Predicción SARIMA ---
df_all_prediction_metrics_sarima_detailed = pd.DataFrame(all_prediction_metrics_sarima_detailed)

if not df_all_prediction_metrics_sarima_detailed.empty:
    detailed_output_path_s = os.path.join(PREDICTION_METRICS_SARIMA_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME_SARIMA}.csv')
    df_all_prediction_metrics_sarima_detailed.to_csv(detailed_output_path_s, index=False)
    print(f"\nTodas las métricas de predicción SARIMA detalladas guardadas en: {detailed_output_path_s}")
    print(df_all_prediction_metrics_sarima_detailed.head())

    # Crear y guardar CSV de métricas promedio por ciudad
    df_summary_by_city_scenario_horizon_s = df_all_prediction_metrics_sarima_detailed.groupby(
        ['City', 'Dataset_Type', 'Scenario', 'Horizon_h']
    ).agg(
        Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'),
        Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Std_MASE=('MASE', 'std'),
        Count_Repetitions=('MAE', 'count')
    ).reset_index()

    for city_name_unique_s in df_summary_by_city_scenario_horizon_s['City'].unique():
        city_name_s_file = city_name_unique_s.replace(" ", "_").replace("-", "_")
        df_city_summary_s = df_summary_by_city_scenario_horizon_s[df_summary_by_city_scenario_horizon_s['City'] == city_name_unique_s]
        
        city_metrics_dir_s = os.path.join(PREDICTION_METRICS_SARIMA_ROOT_DIR, city_name_s_file)
        os.makedirs(city_metrics_dir_s, exist_ok=True)
        city_summary_file_path_s = os.path.join(city_metrics_dir_s, f'metrics_summary_{PREDICTION_METHOD_NAME_SARIMA}_{city_name_s_file}.csv')
        
        df_city_summary_s.to_csv(city_summary_file_path_s, index=False)
        print(f"  Resumen de métricas SARIMA para {city_name_unique_s} guardado en: {city_summary_file_path_s}")

    # Crear y guardar CSV de métricas globales promediadas
    df_global_summary_by_dataset_horizon_s = df_all_prediction_metrics_sarima_detailed.groupby(
        ['Dataset_Type', 'Horizon_h']
    ).agg(
        Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')
    ).reset_index()
    
    global_summary_file_path_s = os.path.join(PREDICTION_METRICS_SARIMA_ROOT_DIR, f'ALL_prediction_metrics_global_summary_{PREDICTION_METHOD_NAME_SARIMA}.csv')
    df_global_summary_by_dataset_horizon_s.to_csv(global_summary_file_path_s, index=False)
    print(f"\nResumen global de métricas SARIMA promedio guardado en: {global_summary_file_path_s}")
    print(df_global_summary_by_dataset_horizon_s)

script_end_time_sarima_pred = time.time()
print(f"\nScript de predicción SARIMA completado. Tiempo total: {(script_end_time_sarima_pred - script_start_time_sarima_pred)/60:.2f} minutos.")